# Cortex Agent — Multi-Tool Orchestration

## Use Case Overview

A **Cortex Agent** is an AI agent that can reason, plan, and take actions using multiple tools — then synthesise results into a coherent answer. Unlike a single `CORTEX.COMPLETE` call, an agent can:
- Decide *which tools to use* based on the question
- Call tools in sequence or parallel
- Pass results from one tool as input to another
- Maintain conversation context across multiple turns

**SA use case:** Build a single conversational assistant that can answer questions about structured sales data (Cortex Analyst), search company knowledge (Cortex Search), and process documents (AI functions) — without the user knowing which backend is handling each question.

**Architecture:**
```
User question
    → Cortex Agent (LLM reasoning layer)
        ├── Tool: Cortex Analyst → SQL queries on GENAI_LEARNING sales data
        ├── Tool: Cortex Search → semantic search over wiki_articles
        └── Tool: SQL Exec → direct queries for structured lookups
```

**Prerequisites:** All phases 1–3 must have been run (this example reuses their tables and services).

## Step 1 — Setup & Connect

In [ ]:
import os
import json
import requests
import pandas as pd
import snowflake.connector

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "genai-labs"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")

HOST = conn.host
TOKEN = conn.rest.token
AGENT_ENDPOINT = f"https://{HOST}/api/v2/cortex/agent:run"

print(f"Account: {conn.account}")
print(f"Agent endpoint: {AGENT_ENDPOINT}")

## Step 2 — Define the Agent Tools

Cortex Agent tools are declared in the request payload. Each tool tells the agent what capability it has and how to call it.

We register two tools:
1. **`analyst_tool`** — wraps the Cortex Analyst semantic model for sales Q&A
2. **`search_tool`** — wraps the Cortex Search Service for knowledge retrieval

In [ ]:
TOOLS = [
    {
        "tool_spec": {
            "type": "cortex_analyst_text_to_sql",
            "name": "sales_analyst",
        },
        "tool_choice": "auto",
    },
    {
        "tool_spec": {
            "type": "cortex_search",
            "name": "wiki_knowledge",
        },
        "tool_choice": "auto",
    },
]

TOOL_RESOURCES = {
    "sales_analyst": {
        "semantic_model_file": "@GENAI_LEARNING.PUBLIC.semantic_models/sales_semantic_model.yaml"
    },
    "wiki_knowledge": {
        "name": "GENAI_LEARNING.PUBLIC.WIKI_SEARCH",
        "max_results": 3,
        "id_column": "article_id",
    },
}

print("Tools defined:")
for t in TOOLS:
    print(f"  - {t['tool_spec']['name']} ({t['tool_spec']['type']})")

## Step 3 — Build the Agent Client

The agent API uses server-sent events (SSE) for streaming responses. The helper below collects the full response and parses tool calls and text output.

In [ ]:
def run_agent(question: str, conversation_history: list = None) -> dict:
    messages = conversation_history or []
    messages.append({"role": "user", "content": question})

    payload = {
        "model": "claude-3-5-sonnet",
        "messages": messages,
        "tools": TOOLS,
        "tool_resources": TOOL_RESOURCES,
        "max_tokens": 4096,
    }

    response = requests.post(
        AGENT_ENDPOINT,
        json=payload,
        headers={
            "Authorization": f"Snowflake Token=\"{TOKEN}\"",
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        stream=True,
        timeout=120,
    )
    response.raise_for_status()

    full_text = ""
    tool_calls_seen = []
    sql_queries = []

    for line in response.iter_lines():
        if not line:
            continue
        line_str = line.decode("utf-8")
        if line_str.startswith("data: "):
            data_str = line_str[6:]
            if data_str == "[DONE]":
                break
            try:
                event = json.loads(data_str)
                delta = event.get("choices", [{}])[0].get("delta", {})
                if delta.get("content"):
                    full_text += delta["content"]
                tool_use = delta.get("tool_use")
                if tool_use:
                    tool_calls_seen.append(tool_use)
                    if tool_use.get("input", {}).get("query"):
                        sql_queries.append(tool_use["input"]["query"])
            except json.JSONDecodeError:
                pass

    messages.append({"role": "assistant", "content": full_text})
    return {
        "answer": full_text,
        "tool_calls": tool_calls_seen,
        "sql_generated": sql_queries,
        "history": messages,
    }


def display_agent_response(result: dict, show_tools: bool = True):
    if show_tools and result["tool_calls"]:
        print("Tools used:")
        for tc in result["tool_calls"]:
            print(f"  → {tc.get('name', 'unknown')}")
        if result["sql_generated"]:
            for sql in result["sql_generated"]:
                print(f"\nGenerated SQL:\n{sql}")
        print()
    print("Agent answer:")
    print(result["answer"])

## Step 4 — Demo 1: Structured Data Question → Cortex Analyst Tool

Ask a business question about sales. The agent should recognise this requires structured data and invoke the `sales_analyst` tool.

In [ ]:
result1 = run_agent("What were the top 5 market segments by total revenue last year?")
display_agent_response(result1)

## Step 5 — Demo 2: Knowledge Question → Search Tool

Ask a conceptual question. The agent should recognise this needs knowledge retrieval and invoke `wiki_knowledge`.

In [ ]:
result2 = run_agent("Can you explain what a vector database is and why it is important for AI applications?")
display_agent_response(result2)

## Step 6 — Demo 3: Multi-Turn Conversation

The agent maintains conversation history. Follow up on the previous answer.

In [ ]:
result3 = run_agent(
    "How does Snowflake implement this natively?",
    conversation_history=result2["history"].copy()
)
display_agent_response(result3)

## Step 7 — Demo 4: Cross-Tool Question

Ask a question that requires both tools — e.g. combining sales data context with product knowledge.

In [ ]:
result4 = run_agent(
    "Which market segment generates the most revenue, and what does the concept of machine learning pipelines have to do with how we might analyse that segment further?"
)
display_agent_response(result4)

## Step 8 — Interpretation & SA Tips

**How the agent reasons:**
1. Reads the user question and available tool descriptions
2. Decides which tool(s) to call based on the question type
3. Calls tools, receives results
4. Synthesises a final answer combining tool outputs and its own knowledge

**SA tips:**
- The quality of tool descriptions in `tool_spec` directly impacts routing accuracy — be explicit
- Add more tools: `sql_exec` for direct SQL, `python_sandbox` via Snowpark for custom logic
- Log `agent_conversations` to `GENAI_LEARNING.PUBLIC.agent_conversations` for debugging and audit trails
- Cortex Agents are the backbone of **Snowflake Intelligence** — the enterprise AI assistant
- Use `max_tokens` and `model` parameters to balance response quality vs. cost
- For production, deploy the agent behind a Streamlit app or REST endpoint